# Document Scanner — Colab GPU Loss Ablation Launcher

Runs the four-loss ablation (ADR-006, `[REQ-45]`) on a Colab T4:

| Run | Loss | Config |
|---|---|---|
| exp-001 | L-A, MSE | `configs/exp/exp-001_enh_mse.yaml` |
| exp-002 | L-B, L1 | `configs/exp/exp-002_enh_l1.yaml` |
| exp-003 | L-C, L1 + MS-SSIM (alpha=0.84) | `configs/exp/exp-003_enh_l1msssim.yaml` |
| exp-004 | L-D, + Sobel (lambda=0.1) | `configs/exp/exp-004_enh_l1msssim_sobel.yaml` |

**Budget.** 2000 samples/epoch at batch 8 is 250 optimiser steps per epoch, over 40 epochs.
Expect roughly **1-1.5 h per run** and **4-6 h for the suite**. That is an estimate from the
measured CPU generator throughput and T4 step times, not a measurement — read the first epoch's
`epoch_seconds` and extrapolate before committing to the whole suite.

A free Colab session will not always survive that. Every run checkpoints each epoch and
**Step 4 mirrors `runs/` to Drive**, so a dropped session resumes where it stopped rather than
starting over. Steps 5a-5d can be run in separate sessions.

> Do not change `batch_size` between runs. It changes BatchNorm statistics and would confound
> the comparison. It lives in `configs/env/colab_t4.yaml` precisely so all four share it.

### Step 0: Confirm the GPU is actually visible

`train.py` now **raises** if a profile asks for CUDA and CUDA is not usable, instead of quietly
falling back to CPU. That silent fallback is what made the previous ablation worthless.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    raise SystemExit('No CUDA device. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.')

### Step 1: Mount Drive, clone the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/DocEn'):
    !git clone https://github.com/HedieTahmouresi/DocEn.git /content/DocEn
%cd /content/DocEn
!git pull
!git log --oneline -1

### Step 2: Data and dependencies

`data.zip` must contain `clean_scans/`, `backgrounds/`, `real_photos/`, `splits/` and — if you
already generated them — `frozen/`. If `frozen/` is absent the cell rebuilds it.

> The frozen val/test sets are the comparability contract (ADR-003). Regenerate them **only**
> when the generator changes, and never in the middle of an ablation: every earlier run becomes
> incomparable. If you do regenerate, bump `frozen_version` in the experiment configs.

In [ ]:
!pip install -q -r requirements.txt

import os

if not os.path.exists('data/clean_scans'):
    for src in ('/content/drive/MyDrive/data.zip', '/content/data.zip'):
        if os.path.exists(src):
            print('Extracting', src)
            !unzip -q "{src}" -d .
            break
    else:
        raise SystemExit('data.zip not found in Drive or /content.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now (a few minutes).')
    !python -m src.data.freeze

!ls data && ls data/frozen

### Step 3: Sanity ladder — run this before any long run

training-spec §9 and the phase-04 gate. The critical check is overfit-one-batch: if the model
cannot fit a single batch, the bug is in the model, the loss or the data, and no amount of GPU
time will fix it. A 40-epoch run that was never going to work is the most expensive mistake
available on a free-tier GPU.

In [ ]:
!python -m pytest tests/ -q -x --durations=10

### Step 3b: Smoke run — two short epochs, end to end

Confirms the loop, the loader, the checkpointing and the CSV logging work on *this* runtime
before four hours are committed to them. The smoke run directory is deleted afterwards so it
cannot be mistaken for the real run.

In [ ]:
!python train.py --config configs/exp/exp-003_enh_l1msssim.yaml --env colab_t4 --epochs 2 --samples-per-epoch 200
!cat runs/exp-003_enh_l1msssim/metrics.csv
!rm -rf runs/exp-003_enh_l1msssim

### Step 4: Mirror checkpoints to Drive

Colab's local disk vanishes with the session (ADR-001, training-spec §7). Run this once and
`runs/` becomes a symlink into Drive, so every checkpoint is written straight there and a
dropped session can resume.

In [ ]:
import os, shutil

DRIVE_RUNS = '/content/drive/MyDrive/DocEn_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.islink('runs'):
    if os.path.isdir('runs'):
        shutil.rmtree('runs')
    os.symlink(DRIVE_RUNS, 'runs')

print('runs/ ->', os.path.realpath('runs'))
print('existing runs:', sorted(os.listdir('runs')))

### Step 5: The four runs

One cell at a time. If a session dies mid-run, re-open the notebook, re-run Steps 1, 2 and 4,
then add `--resume runs/<run_dir>/checkpoints/last.pt` to the cell that was interrupted.

In [ ]:
# 5a - L-A: MSE
!python train.py --config configs/exp/exp-001_enh_mse.yaml --env colab_t4

In [ ]:
# 5b - L-B: L1
!python train.py --config configs/exp/exp-002_enh_l1.yaml --env colab_t4

In [ ]:
# 5c - L-C: L1 + MS-SSIM (alpha=0.84)
!python train.py --config configs/exp/exp-003_enh_l1msssim.yaml --env colab_t4

In [ ]:
# 5d - L-D: L1 + MS-SSIM + Sobel (lambda=0.1)
!python train.py --config configs/exp/exp-004_enh_l1msssim_sobel.yaml --env colab_t4

### Step 6: Figures and the summary table

Produces the `[REQ-22]` loss curves, the `[REQ-45]` zoomed comparison, per-sample restorations,
and `p04_ablation_summary.json` — PSNR/SSIM per variant against the `[REQ-26]` no-model
baseline. **Validation only**: the synthetic test split stays untouched until Phase 05
(`[CON-07]`), and the ablation winner is selected on validation.

In [ ]:
!python -m scripts.evaluate_ablation
!python -m scripts.save_restored_samples

### Step 7: Package the results

Checkpoints are already in Drive via Step 4. This packages the small artefacts — metrics, the
resolved configs and the figures — for committing back to the repository.

In [ ]:
!zip -r phase04_results.zip outputs/figures/ runs/*/metrics.csv runs/*/metrics.json runs/*/config.yaml
!cp phase04_results.zip /content/drive/MyDrive/

from google.colab import files
files.download('phase04_results.zip')